In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor

# Dataset URL
data_url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/housing.data"

# Column names
column_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']

# Read the dataset from the URL and assign column names
housing_data = pd.read_csv(data_url, header=None, names=column_names, sep=r"\s+")

# Display the first few rows
housing_data.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [2]:
# Checking the dimensions of the dataset
print(f'Data Shape: {housing_data.shape}')

# Checking for missing values
housing_data.isnull().sum()

Data Shape: (506, 14)


,0
CRIM,0
ZN,0
INDUS,0
CHAS,0
NOX,0
RM,0
AGE,0
DIS,0
RAD,0
TAX,0


In [3]:
# Separating features (X) and target (y)
X = housing_data.drop('MEDV', axis=1)  # Features
y = housing_data['MEDV']  # Target variable MEDV

# Splitting the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Displaying the shape of the training and testing sets
print(f"Training data shape: {X_train.shape}, Testing data shape: {X_test.shape}")

Training data shape: (404, 13), Testing data shape: (102, 13)


In [4]:
# Train the required models

# Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Ridge Regression with scaling + alpha tuning
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

ridge_param_grid = {
    "ridge__alpha": [0.001, 0.01, 0.1, 0.3, 0.5, 1, 2, 5, 10, 20, 50, 100]
}

ridge_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

ridge_search.fit(X_train, y_train)
ridge_model = ridge_search.best_estimator_

# Lasso Regression with scaling + alpha tuning
lasso_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(max_iter=20000))
])

lasso_param_grid = {
    "lasso__alpha": [0.0005, 0.001, 0.003, 0.005, 0.01, 0.03, 0.05, 0.1, 0.2, 0.5, 1.0]
}

lasso_search = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=lasso_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

lasso_search.fit(X_train, y_train)
lasso_model = lasso_search.best_estimator_

# Gradient Boosting

gbr_param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.03, 0.05, 0.1],
    "max_depth": [2, 3, 4],
    "subsample": [0.8, 1.0]
}

gbr_search = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_grid=gbr_param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

gbr_search.fit(X_train, y_train)
gbr_model = gbr_search.best_estimator_

print("Models trained successfully")
print("Best Ridge parameters:", ridge_search.best_params_)
print("Best Lasso parameters:", lasso_search.best_params_)
print("Best Gradient Boosting parameters:", gbr_search.best_params_)

Models trained successfully
Best Ridge parameters: {'ridge__alpha': 2}
Best Lasso parameters: {'lasso__alpha': 0.0005}
Best Gradient Boosting parameters: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8}


In [5]:
# Function to evaluate a regression model
def evaluate_model(model_name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)

    return {
        "Model": model_name,
        "MSE": mse,
        "RMSE": rmse,
        "R-square": r2,
        "MAE": mae,
        "Predictions": y_pred
    }

# Evaluate all models
linear_result = evaluate_model("Linear Regression", linear_model, X_test, y_test)
ridge_result = evaluate_model("Ridge Regression", ridge_model, X_test, y_test)
lasso_result = evaluate_model("Lasso Regression", lasso_model, X_test, y_test)
gbr_result = evaluate_model("Gradient Boosting", gbr_model, X_test, y_test)

# Store all results together
all_results = [linear_result, ridge_result, lasso_result, gbr_result]

# Find the best model based on highest R-squared
best_result = max(all_results, key=lambda x: x["R-square"])
best_model_name = best_result["Model"]
best_predictions = best_result["Predictions"]

print("Evaluation completed successfully")
print(f"Best model based on highest R-square: {best_model_name}")

Evaluation completed successfully
Best model based on highest R-square: Gradient Boosting


In [6]:
# Creating a DataFrame to store the comparison results
results = pd.DataFrame([
    {
        "Model": linear_result["Model"],
        "MSE": linear_result["MSE"],
        "RMSE": linear_result["RMSE"],
        "R-square": linear_result["R-square"],
        "MAE": linear_result["MAE"]
    },
    {
        "Model": ridge_result["Model"],
        "MSE": ridge_result["MSE"],
        "RMSE": ridge_result["RMSE"],
        "R-square": ridge_result["R-square"],
        "MAE": ridge_result["MAE"]
    },
    {
        "Model": lasso_result["Model"],
        "MSE": lasso_result["MSE"],
        "RMSE": lasso_result["RMSE"],
        "R-square": lasso_result["R-square"],
        "MAE": lasso_result["MAE"]
    },
    {
        "Model": gbr_result["Model"],
        "MSE": gbr_result["MSE"],
        "RMSE": gbr_result["RMSE"],
        "R-square": gbr_result["R-square"],
        "MAE": gbr_result["MAE"]
    }
])

results = results.round(4)

print("Model Comparison Table:")
display(results)

print("\nBest Model Summary:")
print(f"Best Model: {best_result['Model']}")
print(f"MSE: {best_result['MSE']:.4f}")
print(f"RMSE: {best_result['RMSE']:.4f}")
print(f"R-square Score: {best_result['R-square']:.4f}")
print(f"MAE: {best_result['MAE']:.4f}")

Model Comparison Table:


,Model,MSE,RMSE,R-square,MAE
0,Linear Regression,24.2911,4.9286,0.6688,3.1891
1,Ridge Regression,24.3346,4.9330,0.6682,3.1829
2,Lasso Regression,24.2928,4.9288,0.6687,3.1886
3,Gradient Boosting,6.4138,2.5325,0.9125,1.8258



Best Model Summary:
Best Model: Gradient Boosting
MSE: 6.4138
RMSE: 2.5325
R-square Score: 0.9125
MAE: 1.8258


In [7]:
# Example output using the best model

example_output = pd.DataFrame({
    "Actual MEDV": y_test.values[:10],
    "Predicted MEDV": best_predictions[:10],
    "Absolute Error": np.abs(y_test.values[:10] - best_predictions[:10])
})

example_output = example_output.round(2)

print(f"Best Model Used for Example Output: {best_model_name}")
print(f"R-square Score of Best Model: {best_result['R-square']:.4f} or {best_result['R-square'] * 100:.2f}%")
print("\nSample Predictions:")
display(example_output)

print("\nSingle Example:")
sample_index = 0
print("Input Features:")
display(X_test.iloc[[sample_index]])

print(f"Actual MEDV: {y_test.iloc[sample_index]:.2f}")
print(f"Predicted MEDV by {best_model_name}: {best_predictions[sample_index]:.2f}")
print(f"Absolute Error: {abs(y_test.iloc[sample_index] - best_predictions[sample_index]):.2f}")

Best Model Used for Example Output: Gradient Boosting
R-square Score of Best Model: 0.9125 or 91.25%

Sample Predictions:


,Actual MEDV,Predicted MEDV,Absolute Error
0,23.6,23.75,0.15
1,32.4,31.74,0.66
2,13.6,17.36,3.76
3,22.8,23.78,0.98
4,16.1,17.35,1.25
5,20.0,21.68,1.68
6,17.8,18.64,0.84
7,14.0,15.23,1.23
8,19.6,20.94,1.34
9,16.8,20.64,3.84



Single Example:
Input Features:


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
173,0.09178,0.0,4.05,0,0.51,6.416,84.1,2.6463,5,296.0,16.6,395.5,9.04


Actual MEDV: 23.60
Predicted MEDV by Gradient Boosting: 23.75
Absolute Error: 0.15
